In [ ]:
# ============================================================
# CELL 1: Setup
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q soundfile

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"✅ Ready!")

Mounted at /content/drive
PyTorch: 2.11.0+cu130
CUDA: True — Tesla T4
✅ Ready!


In [ ]:
# ============================================================
# CELL 2: Download LibriSpeech (train-clean-100 + dev + test)
# ============================================================
import os

STORAGE = '/content/librispeech_data'
os.makedirs(STORAGE, exist_ok=True)

subsets = {
    'train-clean-100': 'https://www.openslr.org/resources/12/train-clean-100.tar.gz',
    'dev-clean':       'https://www.openslr.org/resources/12/dev-clean.tar.gz',
    'test-clean':      'https://www.openslr.org/resources/12/test-clean.tar.gz',
}

for name, url in subsets.items():
    tar_file = os.path.join(STORAGE, f"{name}.tar.gz")
    extract_check = os.path.join(STORAGE, 'LibriSpeech', name.replace('-', '/') if 'clean' in name else name)

    print(f"\n📥 {name}...")

    if os.path.exists(os.path.join(STORAGE, 'LibriSpeech')):
        # Check if already extracted by looking for .flac files
        flac_count = sum(1 for r, d, f in os.walk(STORAGE) for fn in f if fn.endswith('.flac'))
        if flac_count > 100:
            print(f"  ⏩ LibriSpeech data already exists ({flac_count} flac files)")
            continue

    # Download
    if not os.path.exists(tar_file):
        print(f"  ⬇️  Downloading from {url}...")
        !wget -q --show-progress -O {tar_file} {url}
    else:
        print(f"  ⏩ Archive already exists")

    # Extract
    print(f"  📦 Extracting...")
    !cd {STORAGE} && tar -xzf {tar_file}

    # Delete archive to save space
    os.remove(tar_file)
    print(f"  🗑️  Deleted archive")

# Verify
print("\n" + "=" * 60)
print("📊 LibriSpeech Download Summary")
print("=" * 60)

ls_base = os.path.join(STORAGE, 'LibriSpeech')
for subset in ['train-clean-100', 'dev-clean', 'test-clean']:
    subset_path = os.path.join(ls_base, subset)
    if os.path.exists(subset_path):
        flac_count = sum(1 for r, d, f in os.walk(subset_path) for fn in f if fn.endswith('.flac'))
        speakers = len([d for d in os.listdir(subset_path) if os.path.isdir(os.path.join(subset_path, d))])
        print(f"  ✅ {subset:20s} | {speakers:3d} speakers | {flac_count:5d} utterances")
    else:
        print(f"  ❌ {subset:20s} | NOT FOUND")

!df -h /content | tail -1


📥 train-clean-100...
  ⬇️  Downloading from https://www.openslr.org/resources/12/train-clean-100.tar.gz...
          /content/   7%[>                   ] 458.39M  13.6MB/s    eta 8m 18s ^C
  📦 Extracting...

gzip: stdin: unexpected end of file
tar: Unexpected EOF in archive
tar: Unexpected EOF in archive
tar: Error is not recoverable: exiting now
  🗑️  Deleted archive

📥 dev-clean...
  ⏩ LibriSpeech data already exists (2104 flac files)

📥 test-clean...
  ⏩ LibriSpeech data already exists (2104 flac files)

📊 LibriSpeech Download Summary
  ✅ train-clean-100      |  19 speakers |  2104 utterances
  ❌ dev-clean            | NOT FOUND
  ❌ test-clean           | NOT FOUND
overlay         113G   48G   65G  43% /


In [ ]:
import os

STORAGE = '/content/librispeech_data'
os.makedirs(STORAGE, exist_ok=True)
tar_file = os.path.join(STORAGE, "train-clean-100.tar.gz")

print("📥 Downloading train-clean-100 (~5.9 GB)...")
!wget -q --show-progress -O {tar_file} https://www.openslr.org/resources/12/train-clean-100.tar.gz

print("📦 Extracting...")
!cd {STORAGE} && tar -xzf {tar_file}
os.remove(tar_file)

train_path = os.path.join(STORAGE, 'LibriSpeech', 'train-clean-100')
flac_count = sum(1 for r, d, f in os.walk(train_path) for fn in f if fn.endswith('.flac'))
print(f"✅ train-clean-100 ready: {flac_count} flac files")

📥 Downloading train-clean-100 (~5.9 GB)...
/content/librispeec 100%[===================>]   5.95G  4.61MB/s    in 14m 9s  
📦 Extracting...
✅ train-clean-100 ready: 28539 flac files


In [ ]:
import os
import random
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
from math import gcd

random.seed(42)
np.random.seed(42)

LIBRISPEECH_DIR = '/content/librispeech_data/LibriSpeech'
OUTPUT_DIR = '/content/Libri2Mix/wav8k/min'
TARGET_SR = 8000

def get_speaker_utterances(subset_dir):
    speakers = {}
    for root, dirs, files in os.walk(subset_dir):
        for f in files:
            if f.endswith('.flac'):
                path = os.path.join(root, f)
                spk_id = os.path.relpath(path, subset_dir).split(os.sep)[0]
                if spk_id not in speakers:
                    speakers[spk_id] = []
                speakers[spk_id].append(path)
    return speakers

def load_resample(path, target_sr=8000):
    audio, sr = sf.read(path)
    if sr != target_sr:
        g = gcd(sr, target_sr)
        audio = resample_poly(audio, target_sr // g, sr // g)
    return audio.astype(np.float32)

def generate_train_mixtures():
    subset_dir = os.path.join(LIBRISPEECH_DIR, 'train-clean-100')
    speakers = get_speaker_utterances(subset_dir)
    speaker_ids = sorted(speakers.keys())

    all_utts = []
    for spk_id in speaker_ids:
        for path in speakers[spk_id]:
            all_utts.append((spk_id, path))
    random.shuffle(all_utts)

    mix_dir = os.path.join(OUTPUT_DIR, 'train-100', 'mix_clean')
    s1_dir  = os.path.join(OUTPUT_DIR, 'train-100', 's1')
    s2_dir  = os.path.join(OUTPUT_DIR, 'train-100', 's2')
    os.makedirs(mix_dir, exist_ok=True)
    os.makedirs(s1_dir, exist_ok=True)
    os.makedirs(s2_dir, exist_ok=True)

    pairs = []
    used = set()
    for i in range(len(all_utts)):
        if i in used:
            continue
        for j in range(i + 1, len(all_utts)):
            if j in used:
                continue
            if all_utts[i][0] != all_utts[j][0]:
                pairs.append((all_utts[i], all_utts[j]))
                used.add(i)
                used.add(j)
                break

    print(f"🎯 Generating {len(pairs)} train-100 mixtures...")
    count = 0
    for idx, ((spk1, path1), (spk2, path2)) in enumerate(pairs):
        s1 = load_resample(path1, TARGET_SR)
        s2 = load_resample(path2, TARGET_SR)

        min_len = min(len(s1), len(s2))
        if min_len < TARGET_SR:
            continue
        s1 = s1[:min_len]
        s2 = s2[:min_len]

        snr_db = random.uniform(-5, 5)
        rms1 = np.sqrt(np.mean(s1 ** 2)) + 1e-8
        rms2 = np.sqrt(np.mean(s2 ** 2)) + 1e-8
        s2_scaled = s2 * (rms1 / (rms2 * (10 ** (snr_db / 20))))

        mixture = s1 + s2_scaled
        max_val = np.max(np.abs(mixture))
        if max_val > 0.9:
            scale = 0.9 / max_val
            mixture *= scale
            s1 *= scale
            s2_scaled *= scale

        name1 = os.path.splitext(os.path.basename(path1))[0]
        name2 = os.path.splitext(os.path.basename(path2))[0]
        fname = f"{name1}_{name2}.wav"

        sf.write(os.path.join(mix_dir, fname), mixture, TARGET_SR, subtype='PCM_16')
        sf.write(os.path.join(s1_dir, fname), s1, TARGET_SR, subtype='PCM_16')
        sf.write(os.path.join(s2_dir, fname), s2_scaled, TARGET_SR, subtype='PCM_16')

        count += 1
        if count % 2000 == 0:
            print(f"  ✅ Generated {count}/{len(pairs)}...")

    print(f"\n🎉 Done! Generated {count} mixtures.")

generate_train_mixtures()

# Back up directly to Google Drive
print("\n💾 Archiving train-100 into Google Drive...")
!tar -cf /content/drive/MyDrive/cocktail_party_project/train-100_wav8k.tar -C /content/Libri2Mix/wav8k/min train-100
print("✅ Saved permanently to Drive: train-100_wav8k.tar")

🎯 Generating 14269 train-100 mixtures...
  ✅ Generated 2000/14269...
  ✅ Generated 4000/14269...
  ✅ Generated 6000/14269...
  ✅ Generated 8000/14269...
  ✅ Generated 10000/14269...
  ✅ Generated 12000/14269...
  ✅ Generated 14000/14269...

🎉 Done! Generated 14269 mixtures.

💾 Archiving train-100 into Google Drive...
✅ Saved permanently to Drive: train-100_wav8k.tar


In [ ]:
%%writefile config.py
"""
config.py — High-Efficiency GPU Configuration
"""
import torch
import os

class Config:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    NUM_WORKERS = 4             # 4 parallel workers for fast audio streaming

    CHECKPOINT_DIR = '/content/drive/MyDrive/cocktail_party_project/checkpoints_v2'

    # Audio
    SAMPLE_RATE = 8000
    CHUNK_DURATION = 4.0
    CHUNK_SAMPLES = int(SAMPLE_RATE * CHUNK_DURATION)  # 32,000 samples
    NUM_SPEAKERS = 2

    # Architecture (Paper settings)
    ENC_DIM = 256               # N
    ENC_KERNEL = 20             # L
    ENC_STRIDE = 10             # L/2
    BN_DIM = 256                # B
    HIDDEN_DIM = 512            # H
    SC_DIM = 256                # Sc
    CONV_KERNEL = 3             # P
    NUM_BLOCKS = 8              # X
    NUM_REPEATS = 3             # R
    NUM_SPEAKERS_OUT = 2        # C
    MASK_ACT = 'relu'

    # GPU-Optimized Training
    BATCH_SIZE = 12             # 3x larger batch to fill GPU VRAM
    USE_AMP = True              # FP16 Automatic Mixed Precision (Tensor Cores)
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 0
    NUM_EPOCHS = 100
    GRAD_CLIP = 5.0
    SCHEDULER_PATIENCE = 3

    PRINT_EVERY = 100           # Print every 100 batches

print("✅ config.py updated: Batch Size = 12 | Workers = 4 | FP16 AMP = Enabled")

Writing config.py


In [ ]:
%%writefile dataset.py
"""
dataset.py — Libri2Mix Dataset with max_samples support
"""
import os
import random
import torch
import soundfile as sf
import numpy as np
from torch.utils.data import Dataset
from config import Config


class Libri2MixDataset(Dataset):
    def __init__(self, split='train-100', max_samples=None):
        self.split = split
        self.chunk_samples = Config.CHUNK_SAMPLES

        if split == 'train-100':
            split_dir = '/content/Libri2Mix/wav8k/min/train-100'
        elif split == 'dev':
            split_dir = '/content/drive/MyDrive/cocktail_party_project/data/libri2mix/Libri2Mix/wav8k/min/dev'
        elif split == 'test':
            split_dir = '/content/drive/MyDrive/cocktail_party_project/data/libri2mix/Libri2Mix/wav8k/min/test'
        else:
            raise ValueError(f"Unknown split: {split}")

        self.mix_dir = os.path.join(split_dir, 'mix_clean')
        self.s1_dir  = os.path.join(split_dir, 's1')
        self.s2_dir  = os.path.join(split_dir, 's2')

        all_files = sorted([f for f in os.listdir(self.mix_dir) if f.endswith('.wav')])

        # Shorten if requested (e.g. 300 for fast validation)
        if max_samples is not None and max_samples < len(all_files):
            # Take evenly spaced samples across the set for an unbiased representation
            step = len(all_files) // max_samples
            self.filenames = all_files[::step][:max_samples]
        else:
            self.filenames = all_files

        print(f"  📂 {split.upper()}: {len(self.filenames)} pairs from {split_dir}")

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]

        mixture, _ = sf.read(os.path.join(self.mix_dir, fname), dtype='float32')
        s1, _      = sf.read(os.path.join(self.s1_dir, fname), dtype='float32')
        s2, _      = sf.read(os.path.join(self.s2_dir, fname), dtype='float32')

        min_len = min(len(mixture), len(s1), len(s2))
        mixture, s1, s2 = mixture[:min_len], s1[:min_len], s2[:min_len]

        mixture, s1, s2 = self._chunk(mixture, s1, s2)

        max_val = np.max(np.abs(mixture)) + 1e-8
        mixture /= max_val
        s1 /= max_val
        s2 /= max_val

        mixture = torch.from_numpy(mixture).float()
        sources = torch.stack([
            torch.from_numpy(s1).float(),
            torch.from_numpy(s2).float()
        ])
        return mixture, sources

    def _chunk(self, mixture, s1, s2):
        length = len(mixture)
        if length >= self.chunk_samples:
            if 'train' in self.split:
                start = random.randint(0, length - self.chunk_samples)
            else:
                start = (length - self.chunk_samples) // 2
            end = start + self.chunk_samples
            return mixture[start:end], s1[start:end], s2[start:end]
        else:
            pad = self.chunk_samples - length
            return np.pad(mixture, (0, pad)), np.pad(s1, (0, pad)), np.pad(s2, (0, pad))


print("✅ dataset.py updated with max_samples support")

Writing dataset.py


In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
from config import Config


class Encoder(nn.Module):
    def __init__(self, N, L, stride):
        super().__init__()
        self.conv = nn.Conv1d(1, N, kernel_size=L, stride=stride, bias=False)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.conv(x.unsqueeze(1)))


class Decoder(nn.Module):
    def __init__(self, N, L, stride):
        super().__init__()
        self.deconv = nn.ConvTranspose1d(N, 1, kernel_size=L, stride=stride, bias=False)

    def forward(self, x):
        return self.deconv(x).squeeze(1)


class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, hidden_ch, kernel, dilation, skip_ch):
        super().__init__()
        self.prelu1 = nn.PReLU()
        self.norm1 = nn.GroupNorm(1, in_ch)
        self.dconv = nn.Conv1d(in_ch, in_ch, kernel, dilation=dilation,
                               padding=(kernel - 1) * dilation // 2,
                               groups=in_ch, bias=False)
        self.prelu2 = nn.PReLU()
        self.norm2 = nn.GroupNorm(1, in_ch)
        self.skip_conv = nn.Conv1d(in_ch, skip_ch, 1, bias=False)
        self.res_conv = nn.Conv1d(in_ch, hidden_ch, 1, bias=False)

    def forward(self, x):
        y = self.prelu1(self.norm1(x))
        y = self.dconv(y)
        y = self.prelu2(self.norm2(y))
        skip = self.skip_conv(y)
        res = self.res_conv(y) + x
        return res, skip


class TemporalConvNet(nn.Module):
    def __init__(self, N, B, H, Sc, P, X, R, C, mask_act):
        super().__init__()
        self.C, self.N = C, N
        self.layer_norm = nn.GroupNorm(1, N)
        self.bottleneck = nn.Conv1d(N, B, 1, bias=False)

        self.blocks = nn.ModuleList()
        for r in range(R):
            for x in range(X):
                self.blocks.append(DepthwiseSeparableConv(B, B, P, 2**x, Sc))

        self.output_conv = nn.Conv1d(Sc, N * C, 1, bias=False)
        self.mask_act = nn.ReLU() if mask_act == 'relu' else nn.Sigmoid()

    def forward(self, x):
        B_size, _, K = x.shape
        y = self.bottleneck(self.layer_norm(x))
        skip_sum = 0
        for block in self.blocks:
            y, skip = block(y)
            skip_sum = skip_sum + skip
        masks = self.output_conv(skip_sum)
        masks = masks.view(B_size, self.C, self.N, K)
        return self.mask_act(masks)


class ConvTasNet(nn.Module):
    def __init__(self, enable_mixture_consistency=True):
        super().__init__()
        N, L, S = Config.ENC_DIM, Config.ENC_KERNEL, Config.ENC_STRIDE
        self.encoder = Encoder(N, L, S)
        self.separator = TemporalConvNet(
            N, Config.BN_DIM, Config.HIDDEN_DIM, Config.SC_DIM,
            Config.CONV_KERNEL, Config.NUM_BLOCKS, Config.NUM_REPEATS,
            Config.NUM_SPEAKERS_OUT, Config.MASK_ACT)
        self.decoder = Decoder(N, L, S)
        self.enable_mixture_consistency = enable_mixture_consistency

    def forward(self, mixture):
        enc = self.encoder(mixture)
        masks = self.separator(enc)
        est = []
        for c in range(Config.NUM_SPEAKERS_OUT):
            est.append(self.decoder(enc * masks[:, c]))
        est = torch.stack(est, dim=1)
        est = est[..., :mixture.shape[-1]]

        # ─── UPGRADE: Differentiable Mixture Consistency Layer ───
        # Mathematically guarantees: est_s1 + est_s2 = mixture
        # Eliminates robotic phase voids and volume drift!
        if self.enable_mixture_consistency:
            residual = (mixture.unsqueeze(1) - est.sum(dim=1, keepdim=True)) / Config.NUM_SPEAKERS_OUT
            est = est + residual

        return est

print("✅ model.py updated with Differentiable Mixture Consistency")

Writing model.py


In [ ]:
%%writefile utils.py
import torch
import torch.nn as nn
import itertools


def si_sdr(estimated, target, eps=1e-5):
    estimated = estimated - estimated.mean(dim=-1, keepdim=True)
    target = target - target.mean(dim=-1, keepdim=True)
    dot = torch.sum(estimated * target, dim=-1, keepdim=True)
    s_energy = torch.sum(target ** 2, dim=-1, keepdim=True) + eps
    proj = dot * target / s_energy
    noise = estimated - proj
    return 10 * torch.log10(
        torch.sum(proj ** 2, dim=-1) / (torch.sum(noise ** 2, dim=-1) + eps) + eps)


class MultiResolutionSTFTLoss(nn.Module):
    """
    Computes spectral convergence and log magnitude across 3 FFT resolutions.
    Directly forces the model to learn natural human harmonics without metallic buzz.
    """
    def __init__(self, fft_sizes=[512, 256, 128], hop_sizes=[128, 64, 32], win_lengths=[512, 256, 128]):
        super().__init__()
        self.fft_sizes = fft_sizes
        self.hop_sizes = hop_sizes
        self.win_lengths = win_lengths

    def forward(self, x, y):
        total_loss = 0.0
        for n_fft, hop, win in zip(self.fft_sizes, self.hop_sizes, self.win_lengths):
            window = torch.hann_window(win).to(x.device)
            X = torch.stft(x.float(), n_fft=n_fft, hop_length=hop, win_length=win, window=window, return_complex=True).abs()
            Y = torch.stft(y.float(), n_fft=n_fft, hop_length=hop, win_length=win, window=window, return_complex=True).abs()

            sc_loss = torch.norm(Y - X, p='fro') / (torch.norm(Y, p='fro') + 1e-8)
            log_mag_loss = torch.mean(torch.abs(torch.log(Y + 1e-5) - torch.log(X + 1e-5)))
            total_loss += (sc_loss + log_mag_loss)
        return total_loss / len(self.fft_sizes)


mr_stft_criterion = MultiResolutionSTFTLoss()


def pit_joint_loss(est_sources, target_sources, stft_weight=0.5):
    """
    Permutation Invariant Training over Joint Loss:
    Loss = -SI-SDR + 0.5 * Multi-Resolution STFT Loss
    """
    B, C, T = est_sources.shape
    perms = list(itertools.permutations(range(C)))
    perm_losses = []
    perm_sisdrs = []

    for perm in perms:
        sisdr_val = sum(si_sdr(est_sources[:, c], target_sources[:, p]) for c, p in enumerate(perm)) / C
        stft_val  = sum(mr_stft_criterion(est_sources[:, c], target_sources[:, p]) for c, p in enumerate(perm)) / C

        # Combined objective: maximize SI-SDR, minimize STFT spectral error
        total_loss = -sisdr_val + stft_weight * stft_val
        perm_losses.append(total_loss)
        perm_sisdrs.append(sisdr_val)

    perm_losses = torch.stack(perm_losses, dim=0)
    perm_sisdrs = torch.stack(perm_sisdrs, dim=0)

    # Best permutation assignment
    min_loss, min_idx = perm_losses.min(dim=0)
    best_sisdr = perm_sisdrs.gather(0, min_idx.unsqueeze(0)).squeeze(0)

    return min_loss.mean(), best_sisdr.mean().item()

print("✅ utils.py updated with Multi-Resolution STFT Loss")

Writing utils.py


In [ ]:
%%writefile train.py
import os, time, torch
from torch.utils.data import DataLoader
from config import Config
from dataset import Libri2MixDataset
from model import ConvTasNet
from utils import pit_loss


def train():
    print(f"Device: {Config.DEVICE} ({torch.cuda.get_device_name(0)})")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
    print("=" * 60)

    print("Loading datasets...")
    train_ds = Libri2MixDataset(split='train-100')
    val_ds   = Libri2MixDataset(split='dev', max_samples=300)

    train_loader = DataLoader(
        train_ds, batch_size=Config.BATCH_SIZE,
        shuffle=True, num_workers=Config.NUM_WORKERS,
        pin_memory=True, drop_last=True, persistent_workers=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=Config.BATCH_SIZE,
        shuffle=False, num_workers=Config.NUM_WORKERS,
        pin_memory=True
    )

    model = ConvTasNet().to(Config.DEVICE)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Model: Conv-TasNet | Parameters: {total_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=Config.LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=Config.SCHEDULER_PATIENCE, factor=0.5)

    # Mixed Precision Scaler for T4 Tensor Cores
    scaler = torch.amp.GradScaler('cuda', enabled=Config.USE_AMP)

    os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
    ckpt_path = os.path.join(Config.CHECKPOINT_DIR, 'tasnet_v2_best.pth')
    start_epoch = 1
    best_val = float('-inf')

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=Config.DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_val = ckpt['val_sisdr']
        print(f"  ✅ Resumed from Epoch {ckpt['epoch']} (Best: {best_val:+.2f} dB)")

    print("=" * 60)
    print(f"  Train: {len(train_ds)} pairs | Val: {len(val_ds)} pairs")
    print(f"  Batches/epoch: {len(train_loader)} | Batch size: {Config.BATCH_SIZE}")
    print(f"  Mixed Precision (AMP): {Config.USE_AMP}")
    print("=" * 60)

    for epoch in range(start_epoch, Config.NUM_EPOCHS + 1):
        t0 = time.time()

        # TRAIN
        model.train()
        train_loss, train_sisdr, n = 0, 0, 0
        for batch_idx, (mix, src) in enumerate(train_loader):
            mix = mix.to(Config.DEVICE, non_blocking=True)
            src = src.to(Config.DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            # Accelerated forward pass with AMP
            with torch.amp.autocast('cuda', enabled=Config.USE_AMP):
                est = model(mix)
                loss, sisdr = pit_loss(est, src)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            train_sisdr += sisdr
            n += 1

            if (batch_idx + 1) % Config.PRINT_EVERY == 0:
                print(f"  Epoch {epoch:03d} | Batch {batch_idx+1}/{len(train_loader)} | "
                      f"SI-SDR: {train_sisdr/n:+.2f} dB")

        train_sisdr /= n

        # VALIDATION (300 samples)
        model.eval()
        val_sisdr, vn = 0, 0
        with torch.no_grad():
            for mix, src in val_loader:
                mix = mix.to(Config.DEVICE, non_blocking=True)
                src = src.to(Config.DEVICE, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=Config.USE_AMP):
                    est = model(mix)
                    _, sisdr = pit_loss(est, src)
                val_sisdr += sisdr
                vn += 1
        val_sisdr /= vn

        elapsed = time.time() - t0
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch:03d}/{Config.NUM_EPOCHS}] ({elapsed:.0f}s) | "
              f"Train: {train_sisdr:+.2f} dB | Val: {val_sisdr:+.2f} dB | LR: {lr:.1e}")

        scheduler.step(val_sisdr)

        if val_sisdr > best_val:
            best_val = val_sisdr
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_sisdr': val_sisdr,
                'train_sisdr': train_sisdr,
            }, ckpt_path)
            print(f"  💾 Best model saved! (Val: {val_sisdr:+.2f} dB)")

        if epoch % 5 == 0:
            periodic_path = os.path.join(Config.CHECKPOINT_DIR, f'tasnet_v2_epoch{epoch}.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_sisdr': val_sisdr,
            }, periodic_path)
            print(f"  📌 Checkpoint saved (Epoch {epoch})")

    print(f"\n{'='*60}")
    print(f"🏁 Training complete! Best Val SI-SDR: {best_val:+.2f} dB")
    print(f"   Checkpoint: {ckpt_path}")


if __name__ == '__main__':
    train()

Writing train.py


In [ ]:
%%writefile evaluate.py
import os, torch, numpy as np, soundfile as sf
from config import Config
from model import ConvTasNet
from utils import si_sdr

def evaluate_full_audio():
    model = ConvTasNet().to(Config.DEVICE)
    ckpt_path = os.path.join(Config.CHECKPOINT_DIR, 'tasnet_v2_best.pth')
    ckpt = torch.load(ckpt_path, map_location=Config.DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print(f"✅ Loaded: Epoch {ckpt['epoch']}, Val SI-SDR: {ckpt['val_sisdr']:+.2f} dB")

    test_dir = '/content/drive/MyDrive/cocktail_party_project/data/libri2mix/Libri2Mix/wav8k/min/test'
    mix_dir = os.path.join(test_dir, 'mix_clean')
    filenames = sorted([f for f in os.listdir(mix_dir) if f.endswith('.wav')])

    all_improvements = []
    with torch.no_grad():
        for idx, fname in enumerate(filenames):
            mix, _ = sf.read(os.path.join(test_dir, 'mix_clean', fname), dtype='float32')
            s1, _  = sf.read(os.path.join(test_dir, 's1', fname), dtype='float32')
            s2, _  = sf.read(os.path.join(test_dir, 's2', fname), dtype='float32')

            # Use full audio length, no chunking!
            min_len = min(len(mix), len(s1), len(s2))
            mix, s1, s2 = mix[:min_len], s1[:min_len], s2[:min_len]

            mix_t = torch.from_numpy(mix).float().unsqueeze(0).to(Config.DEVICE)
            s1_t = torch.from_numpy(s1).float().unsqueeze(0).to(Config.DEVICE)
            s2_t = torch.from_numpy(s2).float().unsqueeze(0).to(Config.DEVICE)

            with torch.amp.autocast('cuda', enabled=Config.USE_AMP):
                est = model(mix_t)

            p1 = (si_sdr(est[:,0], s1_t).item() + si_sdr(est[:,1], s2_t).item()) / 2
            p2 = (si_sdr(est[:,0], s2_t).item() + si_sdr(est[:,1], s1_t).item()) / 2
            best = max(p1, p2)

            mb = (si_sdr(mix_t, s1_t).item() + si_sdr(mix_t, s2_t).item()) / 2
            all_improvements.append(best - mb)

            if (idx + 1) % 100 == 0:
                print(f"  Sample {idx+1}/{len(filenames)} | Avg SI-SDRi: {np.mean(all_improvements):+.2f} dB")

    print(f"\n📊 FINAL FULL-AUDIO TEST RESULTS: {np.mean(all_improvements):+.2f} dB")

if __name__ == '__main__':
    evaluate_full_audio()

Writing evaluate.py


In [ ]:
import os
os.makedirs('/content/Libri2Mix/wav8k/min', exist_ok=True)

# Extract from Drive directly to local disk
!tar -xf /content/drive/MyDrive/cocktail_party_project/train-100_wav8k.tar -C /content/Libri2Mix/wav8k/min/

print("✅ Extracted! Checking files:")
!ls /content/Libri2Mix/wav8k/min/train-100/mix_clean | wc -l

✅ Extracted! Checking files:
14269


In [ ]:
!python train.py

✅ config.py updated: Batch Size = 12 | Workers = 4 | FP16 AMP = Enabled
✅ dataset.py updated with max_samples support
✅ model.py created
Device: cuda (Tesla T4)
CUDA Capability: (7, 5)
Loading datasets...
  📂 TRAIN-100: 14269 pairs from /content/Libri2Mix/wav8k/min/train-100
  📂 DEV: 300 pairs from /content/drive/MyDrive/cocktail_party_project/data/libri2mix/Libri2Mix/wav8k/min/dev
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  Model: Conv-TasNet | Parameters: 3,396,144
  ✅ Resumed from Epoch 50 (Best: +11.79 dB)
  Train: 14269 pairs | Val: 300 pairs
  Batches

In [ ]:
!pip install -q pesq pystoi

  Preparing metadata (setup.py) ... done


In [ ]:
%%writefile evaluate.py
import os
import torch
import numpy as np
import soundfile as sf
from torch.utils.data import DataLoader
from config import Config
from model import ConvTasNet
from dataset import Libri2MixDataset
from utils import si_sdr

# Perceptual metrics
from pesq import pesq
from pystoi import stoi

def compute_pesq_safe(ref, deg, sr=8000):
    """Compute Narrowband PESQ safely (handles silent/edge cases)."""
    try:
        # PESQ requires float32 or float64 numpy arrays
        return pesq(sr, ref.astype(np.float32), deg.astype(np.float32), 'nb')
    except Exception:
        return None

def compute_stoi_safe(ref, deg, sr=8000):
    """Compute STOI intelligibility score."""
    try:
        return stoi(ref.astype(np.float32), deg.astype(np.float32), sr, extended=False)
    except Exception:
        return None

def evaluate():
    print("=" * 65)
    print("🚀 COMPREHENSIVE TEST BENCHMARK (SI-SDR + PESQ + STOI)")
    print("=" * 65)

    # 1. Load Model
    ckpt_path = os.path.join(Config.CHECKPOINT_DIR, 'tasnet_v2_best.pth')
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {ckpt_path}")

    model = ConvTasNet().to(Config.DEVICE)
    ckpt = torch.load(ckpt_path, map_location=Config.DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    print(f"✅ Loaded checkpoint: Epoch {ckpt.get('epoch', 'N/A')}")
    print(f"   Validation SI-SDR at save: {ckpt.get('val_sisdr', 0):+.2f} dB")
    print(f"   Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

    # 2. Test Dataset (Evaluate 200 samples with 2 workers to avoid warnings)
    NUM_EVAL_SAMPLES = 200
    test_ds = Libri2MixDataset(split='test', max_samples=NUM_EVAL_SAMPLES)
    test_loader = DataLoader(
        test_ds, batch_size=Config.BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True
    )

    output_audio_dir = os.path.join(Config.CHECKPOINT_DIR, 'audio_samples')
    os.makedirs(output_audio_dir, exist_ok=True)

    # Metric Trackers
    mix_sisdrs, est_sisdrs, sisdr_improvements = [], [], []
    mix_pesqs, est_pesqs = [], []
    mix_stois, est_stois = [], []

    saved_samples_count = 0
    MAX_AUDIO_SAMPLES_TO_SAVE = 5
    sr = Config.SAMPLE_RATE

    print(f"\n🔍 Evaluating {NUM_EVAL_SAMPLES} held-out test samples...")

    with torch.no_grad():
        for batch_idx, (mix, src) in enumerate(test_loader):
            mix = mix.to(Config.DEVICE)
            src = src.to(Config.DEVICE)

            with torch.amp.autocast('cuda', enabled=Config.USE_AMP):
                est = model(mix)

            # Convert model outputs from FP16 back to standard FP32
            est = est.float()
            mix_f = mix.float()
            src_f = src.float()

            B = mix.shape[0]
            for b in range(B):
                m_t = mix_f[b]
                s1_t = src_f[b, 0]
                s2_t = src_f[b, 1]
                e1_t = est[b, 0]
                e2_t = est[b, 1]

                # Permutation matching based on SI-SDR
                p1 = (si_sdr(e1_t, s1_t).item() + si_sdr(e2_t, s2_t).item()) / 2
                p2 = (si_sdr(e1_t, s2_t).item() + si_sdr(e2_t, s1_t).item()) / 2

                if p1 >= p2:
                    best_s1, best_s2 = e1_t, e2_t
                    best_sdr = p1
                else:
                    best_s1, best_s2 = e2_t, e1_t
                    best_sdr = p2

                # Input Baseline SI-SDR
                base_sdr = (si_sdr(m_t, s1_t).item() + si_sdr(m_t, s2_t).item()) / 2
                imprv_sdr = best_sdr - base_sdr

                mix_sisdrs.append(base_sdr)
                est_sisdrs.append(best_sdr)
                sisdr_improvements.append(imprv_sdr)

                # Convert to standard float32 numpy arrays for sf.write & PESQ
                m_np  = m_t.cpu().numpy().astype(np.float32)
                s1_np = s1_t.cpu().numpy().astype(np.float32)
                s2_np = s2_t.cpu().numpy().astype(np.float32)
                e1_np = best_s1.cpu().numpy().astype(np.float32)
                e2_np = best_s2.cpu().numpy().astype(np.float32)

                # PESQ Computation
                q_mix1 = compute_pesq_safe(s1_np, m_np, sr)
                q_mix2 = compute_pesq_safe(s2_np, m_np, sr)
                q_est1 = compute_pesq_safe(s1_np, e1_np, sr)
                q_est2 = compute_pesq_safe(s2_np, e2_np, sr)

                if q_mix1 is not None and q_mix2 is not None and q_est1 is not None and q_est2 is not None:
                    mix_pesqs.append((q_mix1 + q_mix2) / 2)
                    est_pesqs.append((q_est1 + q_est2) / 2)

                # STOI Computation
                st_mix1 = compute_stoi_safe(s1_np, m_np, sr)
                st_mix2 = compute_stoi_safe(s2_np, m_np, sr)
                st_est1 = compute_stoi_safe(s1_np, e1_np, sr)
                st_est2 = compute_stoi_safe(s2_np, e2_np, sr)

                if st_mix1 is not None and st_mix2 is not None and st_est1 is not None and st_est2 is not None:
                    mix_stois.append((st_mix1 + st_mix2) / 2)
                    est_stois.append((st_est1 + st_est2) / 2)

                # Save first few samples for listening
                if saved_samples_count < MAX_AUDIO_SAMPLES_TO_SAVE:
                    idx_str = f"sample_{saved_samples_count + 1:02d}"
                    sf.write(os.path.join(output_audio_dir, f"{idx_str}_mixture.wav"), m_np, sr)
                    sf.write(os.path.join(output_audio_dir, f"{idx_str}_speaker1_est.wav"), e1_np, sr)
                    sf.write(os.path.join(output_audio_dir, f"{idx_str}_speaker2_est.wav"), e2_np, sr)
                    sf.write(os.path.join(output_audio_dir, f"{idx_str}_speaker1_true.wav"), s1_np, sr)
                    sf.write(os.path.join(output_audio_dir, f"{idx_str}_speaker2_true.wav"), s2_np, sr)
                    saved_samples_count += 1

            if (batch_idx + 1) % 5 == 0:
                print(f"  Processed {len(sisdr_improvements)}/{NUM_EVAL_SAMPLES} samples...")

    # Final Academic Summary Report
    print("\n" + "=" * 65)
    print("📊 FINAL MULTI-METRIC TEST BENCHMARK RESULTS")
    print("=" * 65)
    print(f"Evaluated Test Samples: {len(sisdr_improvements)}")
    print("-" * 65)
    print(f"1. Scale-Invariant Signal-to-Distortion Ratio (SI-SDR):")
    print(f"   • Input Mixture:         {np.mean(mix_sisdrs):+.2f} dB")
    print(f"   • Output Separated:      {np.mean(est_sisdrs):+.2f} dB")
    print(f"   • Mean Improvement (Δ):  {np.mean(sisdr_improvements):+.2f} dB  (Median: {np.median(sisdr_improvements):+.2f} dB)")
    print(f"   • Standard Deviation:    {np.std(sisdr_improvements):.2f} dB")
    print("-" * 65)
    if len(mix_pesqs) > 0:
        print(f"2. Perceptual Evaluation of Speech Quality (PESQ, Narrowband 1.0-4.5):")
        print(f"   • Input Mixture:         {np.mean(mix_pesqs):.2f}")
        print(f"   • Output Separated:      {np.mean(est_pesqs):.2f}")
        print(f"   • Mean Improvement (Δ):  {np.mean(est_pesqs) - np.mean(mix_pesqs):+.2f}")
        print("-" * 65)
    if len(mix_stois) > 0:
        print(f"3. Short-Time Objective Intelligibility (STOI, 0.0-1.0):")
        print(f"   • Input Mixture:         {np.mean(mix_stois):.3f}")
        print(f"   • Output Separated:      {np.mean(est_stois):.3f}")
        print(f"   • Mean Improvement (Δ):  {np.mean(est_stois) - np.mean(mix_stois):+.3f}")
        print("-" * 65)
    print(f"💾 Audio samples saved to: {output_audio_dir}")
    print("=" * 65)

if __name__ == '__main__':
    evaluate()

Overwriting evaluate.py


In [ ]:
!python evaluate.py

✅ config.py updated: Batch Size = 12 | Workers = 4 | FP16 AMP = Enabled
✅ model.py created
✅ dataset.py updated with max_samples support
🚀 COMPREHENSIVE TEST BENCHMARK (SI-SDR + PESQ + STOI)
✅ Loaded checkpoint: Epoch 50
   Validation SI-SDR at save: +11.79 dB
   Model Parameters: 3,396,144
  📂 TEST: 200 pairs from /content/drive/MyDrive/cocktail_party_project/data/libri2mix/Libri2Mix/wav8k/min/test

🔍 Evaluating 200 held-out test samples...
  Processed 60/200 samples...
  Processed 120/200 samples...
  Processed 180/200 samples...

📊 FINAL MULTI-METRIC TEST BENCHMARK RESULTS
Evaluated Test Samples: 200
-----------------------------------------------------------------
1. Scale-Invariant Signal-to-Distortion Ratio (SI-SDR):
   • Input Mixture:         +0.01 dB
   • Output Separated:      +11.07 dB
   • Mean Improvement (Δ):  +11.06 dB  (Median: +12.23 dB)
   • Standard Deviation:    3.78 dB
-----------------------------------------------------------------
2. Perceptual Evaluation of Spe

In [ ]:
import os
import IPython.display as ipd

audio_dir = '/content/drive/MyDrive/cocktail_party_project/checkpoints_v2/audio_samples'
sample_id = "sample_01"  # Change to sample_02, sample_03, etc.

print(f"🎧 Listening to {sample_id}:\n")

print("1️⃣ Input Mixture (Two overlapping speakers):")
ipd.display(ipd.Audio(os.path.join(audio_dir, f"{sample_id}_mixture.wav")))

print("2️⃣ Separated Speaker 1 (Model Output):")
ipd.display(ipd.Audio(os.path.join(audio_dir, f"{sample_id}_speaker1_est.wav")))

print("3️⃣ Separated Speaker 2 (Model Output):")
ipd.display(ipd.Audio(os.path.join(audio_dir, f"{sample_id}_speaker2_est.wav")))

print("4️⃣ True Speaker 1 (Clean Reference):")
ipd.display(ipd.Audio(os.path.join(audio_dir, f"{sample_id}_speaker1_true.wav")))

print("5️⃣ True Speaker 2 (Clean Reference):")
ipd.display(ipd.Audio(os.path.join(audio_dir, f"{sample_id}_speaker2_true.wav")))

🎧 Listening to sample_01:

1️⃣ Input Mixture (Two overlapping speakers):


2️⃣ Separated Speaker 1 (Model Output):


3️⃣ Separated Speaker 2 (Model Output):


4️⃣ True Speaker 1 (Clean Reference):


5️⃣ True Speaker 2 (Clean Reference):


In [ ]:
%%writefile finetune.py
import os, time, torch
from torch.utils.data import DataLoader
from config import Config
from dataset import Libri2MixDataset
from model import ConvTasNet
from utils import pit_joint_loss

def finetune():
    print("=" * 65)
    print("🚀 FINE-TUNING CONV-TASNET: ELIMINATING ROBOTIC TONE")
    print("   Objective: Joint Loss (SI-SDR + Multi-Resolution STFT)")
    print("   Architecture: Differentiable Mixture Consistency Enabled")
    print("=" * 65)

    train_ds = Libri2MixDataset(split='train-100')
    val_ds   = Libri2MixDataset(split='dev', max_samples=300)

    train_loader = DataLoader(
        train_ds, batch_size=Config.BATCH_SIZE, shuffle=True,
        num_workers=Config.NUM_WORKERS, pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=Config.BATCH_SIZE, shuffle=False,
        num_workers=Config.NUM_WORKERS, pin_memory=True
    )

    model = ConvTasNet(enable_mixture_consistency=True).to(Config.DEVICE)

    # ─── Load Existing Epoch 50 Checkpoint ───
    ckpt_path = os.path.join(Config.CHECKPOINT_DIR, 'tasnet_v2_best.pth')
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=Config.DEVICE)
        model.load_state_dict(ckpt['model_state_dict'], strict=False)
        print(f"✅ Loaded base checkpoint (Epoch {ckpt.get('epoch', 50)}, Val SI-SDR: {ckpt.get('val_sisdr', 0):+.2f} dB)")
    else:
        print("⚠️ Checkpoint not found, starting fresh.")

    # Gentle fine-tuning learning rate (1e-4)
    FINETUNE_LR = 1e-4
    NUM_FINETUNE_EPOCHS = 15
    optimizer = torch.optim.Adam(model.parameters(), lr=FINETUNE_LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)
    scaler = torch.amp.GradScaler('cuda', enabled=Config.USE_AMP)

    save_dir = os.path.join(Config.CHECKPOINT_DIR, 'finetuned_clean')
    os.makedirs(save_dir, exist_ok=True)
    best_clean_ckpt = os.path.join(save_dir, 'tasnet_clean_best.pth')

    best_val_sisdr = float('-inf')

    print(f"\nTraining for {NUM_FINETUNE_EPOCHS} fine-tuning epochs at LR = {FINETUNE_LR:.1e}...")
    print("=" * 65)

    for epoch in range(1, NUM_FINETUNE_EPOCHS + 1):
        t0 = time.time()
        model.train()
        train_sisdr, n = 0, 0

        for batch_idx, (mix, src) in enumerate(train_loader):
            mix = mix.to(Config.DEVICE, non_blocking=True)
            src = src.to(Config.DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=Config.USE_AMP):
                est = model(mix)
                loss, sisdr_val = pit_joint_loss(est, src, stft_weight=0.5)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            train_sisdr += sisdr_val
            n += 1

            if (batch_idx + 1) % Config.PRINT_EVERY == 0:
                print(f"  FineTune Ep {epoch:02d} | Batch {batch_idx+1}/{len(train_loader)} | SI-SDR: {train_sisdr/n:+.2f} dB")

        train_sisdr /= n

        # Validation
        model.eval()
        val_sisdr, vn = 0, 0
        with torch.no_grad():
            for mix, src in val_loader:
                mix = mix.to(Config.DEVICE, non_blocking=True)
                src = src.to(Config.DEVICE, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=Config.USE_AMP):
                    est = model(mix)
                    _, sisdr_val = pit_joint_loss(est, src, stft_weight=0.5)
                val_sisdr += sisdr_val
                vn += 1
        val_sisdr /= vn

        elapsed = time.time() - t0
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch:02d}/{NUM_FINETUNE_EPOCHS}] ({elapsed:.0f}s) | Train: {train_sisdr:+.2f} dB | Val: {val_sisdr:+.2f} dB | LR: {lr:.1e}")

        scheduler.step(val_sisdr)

        if val_sisdr > best_val_sisdr:
            best_val_sisdr = val_sisdr
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_sisdr': val_sisdr,
                'train_sisdr': train_sisdr,
            }, best_clean_ckpt)
            print(f"  💾 Best Clean Model saved! (Val SI-SDR: {val_sisdr:+.2f} dB)")

    print("\n" + "=" * 65)
    print(f"🏁 Fine-Tuning Complete! Best Model: {best_clean_ckpt}")

if __name__ == '__main__':
    finetune()

Writing finetune.py


In [ ]:
!python finetune.py

✅ config.py updated: Batch Size = 12 | Workers = 4 | FP16 AMP = Enabled
✅ dataset.py updated with max_samples support
✅ model.py updated with Differentiable Mixture Consistency
✅ utils.py updated with Multi-Resolution STFT Loss
🚀 FINE-TUNING CONV-TASNET: ELIMINATING ROBOTIC TONE
   Objective: Joint Loss (SI-SDR + Multi-Resolution STFT)
   Architecture: Differentiable Mixture Consistency Enabled
  📂 TRAIN-100: 14269 pairs from /content/Libri2Mix/wav8k/min/train-100
  📂 DEV: 300 pairs from /content/drive/MyDrive/cocktail_party_project/data/libri2mix/Libri2Mix/wav8k/min/dev
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freez

In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
from config import Config

class Encoder(nn.Module):
    def __init__(self, N, L, stride):
        super().__init__()
        self.conv = nn.Conv1d(1, N, kernel_size=L, stride=stride, bias=False)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.conv(x.unsqueeze(1)))

class Decoder(nn.Module):
    def __init__(self, N, L, stride):
        super().__init__()
        self.deconv = nn.ConvTranspose1d(N, 1, kernel_size=L, stride=stride, bias=False)

    def forward(self, x):
        return self.deconv(x).squeeze(1)

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, hidden_ch, kernel, dilation, skip_ch):
        super().__init__()
        self.prelu1 = nn.PReLU()
        self.norm1 = nn.GroupNorm(1, in_ch)
        self.dconv = nn.Conv1d(in_ch, in_ch, kernel, dilation=dilation,
                               padding=(kernel - 1) * dilation // 2,
                               groups=in_ch, bias=False)
        self.prelu2 = nn.PReLU()
        self.norm2 = nn.GroupNorm(1, in_ch)
        self.skip_conv = nn.Conv1d(in_ch, skip_ch, 1, bias=False)
        self.res_conv = nn.Conv1d(in_ch, hidden_ch, 1, bias=False)

    def forward(self, x):
        y = self.prelu1(self.norm1(x))
        y = self.dconv(y)
        y = self.prelu2(self.norm2(y))
        skip = self.skip_conv(y)
        res = self.res_conv(y) + x
        return res, skip

class TemporalConvNet(nn.Module):
    def __init__(self, N, B, H, Sc, P, X, R, C, mask_act):
        super().__init__()
        self.C, self.N = C, N
        self.layer_norm = nn.GroupNorm(1, N)
        self.bottleneck = nn.Conv1d(N, B, 1, bias=False)

        self.blocks = nn.ModuleList()
        for r in range(R):
            for x in range(X):
                self.blocks.append(DepthwiseSeparableConv(B, B, P, 2**x, Sc))

        self.output_conv = nn.Conv1d(Sc, N * C, 1, bias=False)
        self.mask_act = nn.ReLU() if mask_act == 'relu' else nn.Sigmoid()

    def forward(self, x):
        B_size, _, K = x.shape
        y = self.bottleneck(self.layer_norm(x))
        skip_sum = 0
        for block in self.blocks:
            y, skip = block(y)
            skip_sum = skip_sum + skip
        masks = self.output_conv(skip_sum)
        masks = masks.view(B_size, self.C, self.N, K)
        return self.mask_act(masks)

class ConvTasNet(nn.Module):
    def __init__(self):
        super().__init__()
        N, L, S = Config.ENC_DIM, Config.ENC_KERNEL, Config.ENC_STRIDE
        self.encoder = Encoder(N, L, S)
        self.separator = TemporalConvNet(
            N, Config.BN_DIM, Config.HIDDEN_DIM, Config.SC_DIM,
            Config.CONV_KERNEL, Config.NUM_BLOCKS, Config.NUM_REPEATS,
            Config.NUM_SPEAKERS_OUT, Config.MASK_ACT)
        self.decoder = Decoder(N, L, S)

    def forward(self, mixture):
        enc = self.encoder(mixture)
        masks = self.separator(enc)
        est = []
        for c in range(Config.NUM_SPEAKERS_OUT):
            est.append(self.decoder(enc * masks[:, c]))
        est = torch.stack(est, dim=1)
        return est[..., :mixture.shape[-1]]

print("✅ model.py restored to clean architecture")

Overwriting model.py


In [ ]:
%%writefile evaluate_clean.py
import os
import torch
import numpy as np
import soundfile as sf
from torch.utils.data import DataLoader
from config import Config
from model import ConvTasNet
from dataset import Libri2MixDataset
from utils import si_sdr

def mixture_consistent_wiener(mix, est1, est2, n_fft=512, hop_length=128):
    """Reconstructs speech preserving the natural human vocal phase."""
    window = torch.hann_window(n_fft).to(mix.device)
    X  = torch.stft(mix,  n_fft=n_fft, hop_length=hop_length, window=window, return_complex=True)
    S1 = torch.stft(est1, n_fft=n_fft, hop_length=hop_length, window=window, return_complex=True)
    S2 = torch.stft(est2, n_fft=n_fft, hop_length=hop_length, window=window, return_complex=True)

    pow1 = torch.abs(S1) ** 2
    pow2 = torch.abs(S2) ** 2
    total_pow = pow1 + pow2 + 1e-8

    M1 = pow1 / total_pow
    M2 = pow2 / total_pow

    clean_S1 = M1 * X
    clean_S2 = M2 * X

    clean_s1 = torch.istft(clean_S1, n_fft=n_fft, hop_length=hop_length, window=window, length=mix.shape[-1])
    clean_s2 = torch.istft(clean_S2, n_fft=n_fft, hop_length=hop_length, window=window, length=mix.shape[-1])
    return clean_s1, clean_s2

def to_clean_numpy(tensor, target_peak=0.90):
    arr = tensor.cpu().numpy().astype(np.float32)
    arr = arr - np.mean(arr)
    peak = np.max(np.abs(arr)) + 1e-8
    return (arr / peak) * target_peak

def run_inference():
    print("=" * 65)
    print("🚀 RUNNING FINAL CLEAN INFERENCE (Epoch 50 + Wiener Decoder)")
    print("=" * 65)

    # 1. Load Original Epoch 50 Checkpoint
    ckpt_path = os.path.join(Config.CHECKPOINT_DIR, 'tasnet_v2_best.pth')
    if not os.path.exists(ckpt_path):
        ckpt_path = os.path.join(Config.CHECKPOINT_DIR, 'tasnet_v2_epoch50.pth')

    model = ConvTasNet().to(Config.DEVICE)
    ckpt = torch.load(ckpt_path, map_location=Config.DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print(f"✅ Loaded: Epoch {ckpt.get('epoch', 50)} (Val SI-SDR: {ckpt.get('val_sisdr', 0):+.2f} dB)")

    # 2. Test Dataset
    NUM_SAMPLES = 50
    test_ds = Libri2MixDataset(split='test', max_samples=NUM_SAMPLES)
    test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)

    output_dir = os.path.join(Config.CHECKPOINT_DIR, 'final_clean_audio')
    os.makedirs(output_dir, exist_ok=True)

    improvements = []
    saved_count = 0
    MAX_SAVE = 5
    sr = Config.SAMPLE_RATE

    with torch.no_grad():
        for batch_idx, (mix, src) in enumerate(test_loader):
            mix = mix.to(Config.DEVICE)
            src = src.to(Config.DEVICE)

            with torch.amp.autocast('cuda', enabled=Config.USE_AMP):
                raw_est = model(mix)

            B = mix.shape[0]
            for b in range(B):
                m   = mix[b].float()
                s1  = src[b, 0].float()
                s2  = src[b, 1].float()
                re1 = raw_est[b, 0].float()
                re2 = raw_est[b, 1].float()

                # Clean natural reconstruction
                clean_s1, clean_s2 = mixture_consistent_wiener(m, re1, re2)

                # SI-SDR Improvement
                p1 = (si_sdr(clean_s1, s1).item() + si_sdr(clean_s2, s2).item()) / 2
                p2 = (si_sdr(clean_s1, s2).item() + si_sdr(clean_s2, s1).item()) / 2

                if p1 >= p2:
                    out_s1, out_s2 = clean_s1, clean_s2
                    best_score = p1
                else:
                    out_s1, out_s2 = clean_s2, clean_s1
                    best_score = p2

                base = (si_sdr(m, s1).item() + si_sdr(m, s2).item()) / 2
                improvements.append(best_score - base)

                # Save 5 sample pairs to Drive
                if saved_count < MAX_SAVE:
                    idx_str = f"pair_{saved_count + 1:02d}"
                    sf.write(os.path.join(output_dir, f"{idx_str}_mixture.wav"), to_clean_numpy(m), sr)
                    sf.write(os.path.join(output_dir, f"{idx_str}_speaker1_clean.wav"), to_clean_numpy(out_s1), sr)
                    sf.write(os.path.join(output_dir, f"{idx_str}_speaker2_clean.wav"), to_clean_numpy(out_s2), sr)
                    sf.write(os.path.join(output_dir, f"{idx_str}_speaker1_target.wav"), to_clean_numpy(s1), sr)
                    sf.write(os.path.join(output_dir, f"{idx_str}_speaker2_target.wav"), to_clean_numpy(s2), sr)
                    saved_count += 1

    print("\n" + "=" * 65)
    print("📊 INFERENCE BENCHMARK COMPLETE")
    print("=" * 65)
    print(f"  Test Samples Evaluated:    {len(improvements)}")
    print(f"  Mean SI-SDRi Improvement:  {np.mean(improvements):+.2f} dB")
    print(f"  Median SI-SDRi:            {np.median(improvements):+.2f} dB")
    print(f"💾 Clean audio files saved to:")
    print(f"   {output_dir}")
    print("=" * 65)

if __name__ == '__main__':
    run_inference()

Writing evaluate_clean.py


In [ ]:
!python evaluate_clean.py

✅ config.py updated: Batch Size = 12 | Workers = 4 | FP16 AMP = Enabled
✅ model.py restored to clean architecture
✅ dataset.py updated with max_samples support
✅ utils.py updated with Multi-Resolution STFT Loss
🚀 RUNNING FINAL CLEAN INFERENCE (Epoch 50 + Wiener Decoder)
✅ Loaded: Epoch 50 (Val SI-SDR: +11.79 dB)
  📂 TEST: 50 pairs from /content/drive/MyDrive/cocktail_party_project/data/libri2mix/Libri2Mix/wav8k/min/test

📊 INFERENCE BENCHMARK COMPLETE
  Test Samples Evaluated:    50
  Mean SI-SDRi Improvement:  +10.43 dB
  Median SI-SDRi:            +11.36 dB
💾 Clean audio files saved to:
   /content/drive/MyDrive/cocktail_party_project/checkpoints_v2/final_clean_audio
